# **Rome Airbnb Clusterization**

**Project**: Airbnb Market Segmentation Analysis

**Goal**: Cluster Rome's Airbnb listings using the final feature set from EDA & Feature Selection, testing systematically whether NLP-derived signals reveal market structure invisible to tabular features alone, resolving the geography input, tuning k and gamma, and validating cluster stability before profiling the resulting segments.

## 1. Setup & Preprocessing

### 1.1 Imports & Environment

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
import html

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [2]:
# Base path configuration
BASE_PATH = Path.cwd().parent

### 1.2 Data Loading

In [3]:
listings = pd.read_csv(BASE_PATH / 'data' / 'processed' / 'listings_final.csv')

In [4]:
listings.drop(columns = ['Unnamed: 0'], inplace = True)
print(f"shape: {listings.shape}")
print(f"\ndtypes:\n{listings.dtypes.value_counts()}")

shape: (35818, 149)

dtypes:
float64    86
str        45
int64      18
Name: count, dtype: int64


### 1.3 Split Features by Type

In [6]:
listing_ids = listings['id'].copy()
price = listings['price']
price_tier = listings['price_tier']

numeric_all = listings.select_dtypes(include=[np.number]).columns.tolist()
str_cols = listings.select_dtypes(include='str').columns.tolist()

EXCLUDE_FROM_NUMERIC = {'id', 'price'}

binary_cols = [c for c in numeric_all if c not in EXCLUDE_FROM_NUMERIC
               and set(listings[c].dropna().unique()).issubset({0, 1})]
continuous_cols = [c for c in numeric_all if c not in binary_cols and c not in EXCLUDE_FROM_NUMERIC]
categorical_cols = [c for c in str_cols if c != 'price_tier']

print(f"binary flags: {len(binary_cols)}")
print(f"continuous numeric: {len(continuous_cols)}")
print(f"string/nominal categorical: {len(categorical_cols)}")
print(f"\nid excluded from clustering inputs: {'id' not in continuous_cols and 'id' not in binary_cols}")
print(f"price excluded from clustering inputs: {'price' not in continuous_cols and 'price' not in binary_cols}")
print(f"price_tier excluded from clustering inputs: {'price_tier' not in categorical_cols}")

# sanity check: every clustering-relevant column accounted for exactly once
all_clustering_cols = binary_cols + continuous_cols + categorical_cols
print(f"\ntotal clustering-input columns: {len(all_clustering_cols)}")
print(f"total listings columns: {listings.shape[1]}")
print(f"expected gap: {listings.shape[1] - len(all_clustering_cols)}")

binary flags: 29
continuous numeric: 73
string/nominal categorical: 44

id excluded from clustering inputs: True
price excluded from clustering inputs: True
price_tier excluded from clustering inputs: True

total clustering-input columns: 146
total listings columns: 149
expected gap: 3


### 1.4 Build the Final Clustering Matrix

In [8]:
### 1.4 Build the Final Clustering Matrix
from sklearn.preprocessing import StandardScaler

X_categorical = listings[categorical_cols + binary_cols].copy()
for col in binary_cols:
    X_categorical[col] = X_categorical[col].astype(int).astype(str)

X_continuous = listings[continuous_cols].copy()
scaler = StandardScaler()
X_continuous_scaled = pd.DataFrame(
    scaler.fit_transform(X_continuous),
    columns=continuous_cols,
    index=listings.index
)

print(f"X_categorical: {X_categorical.shape} ({len(categorical_cols)} nominal + {len(binary_cols)} binary-as-string)")
print(f"X_continuous_scaled: {X_continuous_scaled.shape}")

print(f"\nX_categorical any NaN: {X_categorical.isna().any().any()}")
print(f"X_continuous_scaled any NaN: {X_continuous_scaled.isna().any().any()}")
print(f"X_continuous_scaled mean: {X_continuous_scaled.mean().abs().max():.6f}")
print(f"X_continuous_scaled std: {X_continuous_scaled.std().min():.4f} to {X_continuous_scaled.std().max():.4f}")

X_categorical: (35818, 73) (44 nominal + 29 binary-as-string)
X_continuous_scaled: (35818, 73)

X_categorical any NaN: False
X_continuous_scaled any NaN: False
X_continuous_scaled mean: 0.000000
X_continuous_scaled std: 1.0000 to 1.0000


In [9]:
X_combined = pd.concat([X_continuous_scaled, X_categorical], axis=1)
categorical_indices = list(range(len(continuous_cols), len(continuous_cols) + len(X_categorical.columns)))

print(f"\nX_combined shape: {X_combined.shape}")
print(f"categorical column indices: {categorical_indices[0]} to {categorical_indices[-1]} "
      f"({len(categorical_indices)} categorical columns)")

# sanity check: id, price, price_tier genuinely absent from the clustering matrix
leaked = {'id', 'price', 'price_tier'} & set(X_combined.columns)
print(f"\nleaked external columns in X_combined: {leaked if leaked else 'none — clean'}")

# sanity check: row alignment between listing_ids and X_combined preserved throughout
print(f"listing_ids length: {len(listing_ids)}, X_combined length: {len(X_combined)}")
print(f"indices match: {(listing_ids.index == X_combined.index).all()}")


X_combined shape: (35818, 146)
categorical column indices: 73 to 145 (73 categorical columns)

leaked external columns in X_combined: none — clean
listing_ids length: 35818, X_combined length: 35818
indices match: True


`id`/`price`/`price_tier` were separated from the final feature set as external (kept for later profiling). The rest was split into continuous, binary, and nominal categorical, then scaled the continuous block and assembled the combined matrix `kmodes.KPrototypes` expects. Binary flags travel with the categorical block, as strings, so their weight in the distance is governed by `gamma` rather than `StandardScaler`'s per-column variance.